# Домашнее задание: Pydantic


## Важно!

- При выполнении задания используем точные типы (`EmailStr`, `HttpUrl`, `SecretStr`, `Decimal`, конкретные `Enum`).
- Придерживаемся принципа разделения валидаций: проверка поля — в `field_validator`, сквозные зависимости — в `model_validator`


## Задача 1. Профиль пользователя (валидация полей)

Постройте модель профиля пользователя для внутренней CRM:

**Требования**
1. Обязательные поля: `id: UUID`, `email: EmailStr`, `name: str`.
2. Опциональные поля: `website: HttpUrl | None`, `bio: str | None`.
3. Пароль хранится как `SecretStr`, должен быть не короче 8 символов.
4. Имя (`name`) нормализуйте: тримминг + одна пробельная последовательность между словами + первая буква каждого слова заглавная.
5. Если указан `website`, домен сайта не должен совпадать с доменом `email` (смысл: личный сайт != корпоративная почта).

Подсказки: используйте `field_validator` для нормализации и локальных проверок; и `model_validator(mode="after")` для проверки зависимости `email` ↔ `website`.


In [79]:
!pip install --upgrade gradio
!pip install -U pydantic[email,timezone] -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.2/315.2 kB 21.0 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.13.3
    Uninstalling gradio_client-1.13.3:
      Successfully uninstalled gradio_client-1.13.3
  Attempting uninstall: gradio
    Found existing installation: gradio 5.49.1
    Uninstalling gradio-5.49.1:
      Successfully uninstalled gradio-5.49.1


In [80]:
import logging
import sys

log_format = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
logging.basicConfig(
    level=logging.INFO,
    format=log_format,
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)

logger = logging.getLogger(__name__)

In [81]:
from typing import Optional
from pydantic import BaseModel, Field, EmailStr, HttpUrl, SecretStr
from pydantic import field_validator, model_validator
from uuid import UUID, uuid4

class UserProfile(BaseModel):
    # TODO: опишите поля согласно требованиям
    id: UUID
    email: EmailStr
    name: str
    password: SecretStr
    website: HttpUrl | None = None
    bio: str | None = None

    # TODO: нормализация имени
    @field_validator("name")
    @classmethod
    def normalize_name(cls, v: str) -> str:
        return ' '.join(
          name_part.replace(name_part[0], name_part[0].upper())
          for name_part in v.split()
        )

    # TODO: проверка длины пароля
    @field_validator("password")
    @classmethod
    def password_strength(cls, v: SecretStr) -> SecretStr:
        if len(v) < 8:
          raise Exception("password length should be >=8")

        return v

    # TODO: сквозная проверка доменов email/website
    @model_validator(mode="after")
    def check_domains(self):
        '''
        Про домены не очень понятное условие,
        нужно ли бить по уровням или нет
        поэтому сравниваем полный домен.
        '''
        if self.website:
          if self.website.host == self.email.split('@')[-1]:
            raise Exception('Email domain should not be equal website domain')

        return self



user_profile = UserProfile(
    id=uuid4(),
    email="filipp@mail.com",
    name="    filipp   schulz  ",
    password="ffffffff",
    website="https://filippcv.mail.com",
    bio="Hello, I'm from Russia"
)
logger.info(f"User profile instance: {user_profile}")

try:
  user_profile = UserProfile(
      id=uuid4(),
      email="filipp@mail.com",
      name="    filipp   schulz  ",
      password="ffffffff",
      website="https://mail.com",
      bio="Hello, I'm from Russia"
  )
except Exception as err:
  logger.error(f"during creating model {err}")

2025-11-23 13:42:31,632 - __main__ - INFO - User profile instance: id=UUID('3148bda5-b725-4b82-bd1b-7584c5762f69') email='filipp@mail.com' name='Filipp Schulz' password=SecretStr('**********') website=HttpUrl('https://filippcv.mail.com/') bio="Hello, I'm from Russia"
2025-11-23 13:42:31,634 - __main__ - ERROR - during creating model Email domain should not be equal website domain


## Задача 2. Валидация функции заказа (`@validate_call`)

Реализуйте функцию `place_order`, которая принимает:
- `user_id: UUID`
- `sku: str` (артикул, только заглавные буквы/цифры, длина 3–12)
- `quantity: int` (>0)
- `price: Decimal` (>= 0), округляется банковским методом до 2 знаков

Функция должна возвращать словарь с ключами: `user_id`, `sku`, `quantity`, `price`, `amount` (quantity × price).

Используйте `@validate_call` и локальные проверки через обычный код (или вспомогательные валидаторы `TypeAdapter` не используем).


In [82]:
from pydantic import validate_call
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID
import re

def validate_price(price: Decimal) -> bool:
    return price >= 0

def validate_quantity(quantity: int) -> bool:
    return quantity > 0

def validate_sku(sku: str) -> bool:
    '''
    Вижу что по импортам предлагается использовать re,
    но кажется избыточно для такой лёгкой проверки,
    такой подход будет оптимальнее
    '''
    if sku.isalnum() and (sku.isupper() or sku.isnumeric()) \
      and 3 <= len(sku) <= 12:
        return True
    return False

# TODO: реализуйте функцию с @validate_call
@validate_call
def place_order(user_id: UUID, sku: str, quantity: int, price: Decimal):
    if not validate_sku(sku):
        raise Exception("sku format error")

    if not validate_quantity(quantity):
        raise Exception("quantity should be >0")

    if not validate_price(price):
        raise Exception("price should be >=0")

    rounded_price = price.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)
    amount = quantity * rounded_price

    return {'user_id': user_id, 'sku': sku, 'quantity': quantity,
            'price': rounded_price, 'amount': amount}


exec = place_order(uuid4(), '3FDG33333', 2, Decimal(1.999))
logger.info(f"func place_order result: {exec}")

try:
  exec = place_order(uuid4(), '3333!3', 2, Decimal(1.999))
except Exception as err:
    logger.error(f"during model creating: {err}")

2025-11-23 13:42:40,091 - __main__ - INFO - func place_order result: {'user_id': UUID('36820677-76db-4cd4-986f-1952b436de6b'), 'sku': '3FDG33333', 'quantity': 2, 'price': Decimal('2.00'), 'amount': Decimal('4.00')}
2025-11-23 13:42:40,092 - __main__ - ERROR - during model creating: sku format error


## Задача 3. Модель заказа с бизнес-правилами

Смоделируйте заказ в магазине цифровых товаров.

**Требования**
- `OrderStatus: Enum` со значениями `new`, `paid`, `delivered`, `canceled`.
- Модель `OrderItem`:
  - `sku: str` как в задаче 2
  - `qty: int` (>0)
  - `unit_price: Decimal` (>=0) округление до 2 знаков
- Модель `Order`:
  - `id: UUID`
  - `user_email: EmailStr`
  - `items: list[OrderItem]` (не пустой)
  - `status: OrderStatus = 'new'`
  - `created_at: datetime` (по умолчанию `datetime.utcnow`)
  - Расчитанное поле `total: Decimal` — сумма по всем позициям
  - В `model_validator(mode="after")` запретите переход в `paid`/`delivered` при `total == 0` и запретите пустые корзины.

**Важно:** используйте только инструменты `pydantic` и стандартную библиотеку.


In [83]:
from pydantic import BaseModel, computed_field, EmailStr, field_validator, model_validator
from typing import List
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID
from datetime import datetime
from enum import Enum

SKU_RE = re.compile(r"^[A-Z0-9]{3,12}$")

class OrderStatus(str, Enum):
    # TODO: перечислите статусы
    new = 'new'
    paid = 'paid'
    delivered = 'delivered'
    canceled = 'canceled'

class OrderItem(BaseModel):
    # TODO: опишите поля
    sku: str
    qty: int
    unit_price: Decimal

    @field_validator("sku")
    @classmethod
    def sku_format(cls, v: str) -> str:
        if SKU_RE.fullmatch(v):
          return v

        raise Exception("Bad sku format")


    @field_validator("qty")
    @classmethod
    def qty_positive(cls, v: int) -> int:
        if v > 0:
          return v

        raise Exception("qty should be >0")

    @field_validator("unit_price")
    @classmethod
    def price_non_negative(cls, v: Decimal) -> Decimal:
        if v >= 0:
          return v.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)

        raise Exception("price should be >=0")


class Order(BaseModel):
    # TODO: опишите поля
    id: UUID
    user_email: EmailStr
    items: list[OrderItem]
    status: OrderStatus = 'new'
    created_at: datetime = datetime.utcnow()

    @model_validator(mode="after")
    def check_business_rules(self):
        if self.total == 0 and self.status in (OrderStatus.paid, OrderStatus.delivered):
            raise Exception("the status cannot be equal to paid or delivered if total is equal to zero")

        if not self.items:
            raise Exception("The shopping cart cannot be empty")

        return self

    @computed_field(return_type=Decimal)
    @property
    def total(self):
        return Decimal(sum((item.unit_price * item.qty) for item in self.items))


order_item = OrderItem(sku="AAA", qty=2, unit_price=Decimal(44.444))
order = Order(id=uuid4(), status='new', user_email="filipp@mail.com",
              items=[order_item])
logger.info(f"Order instance example: {order}")


order_item = OrderItem(sku="AAA", qty=2, unit_price=Decimal(0))
try:
  order = Order(id=uuid4(), status='paid', user_email="filipp@mail.com",
                items=[order_item])
except Exception as err:
  logger.error(f"error during model creating: {err}")

2025-11-23 13:42:48,056 - __main__ - INFO - Order instance example: id=UUID('4dad1c81-857a-43fc-b4fd-c72409910f2c') user_email='filipp@mail.com' items=[OrderItem(sku='AAA', qty=2, unit_price=Decimal('44.44'))] status=<OrderStatus.new: 'new'> created_at=datetime.datetime(2025, 11, 23, 13, 42, 48, 51540) total=Decimal('88.88')
2025-11-23 13:42:48,057 - __main__ - ERROR - error during model creating: the status cannot be equal to paid or delivered if total is equal to zero


/tmp/ipython-input-1731136558.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at: datetime = datetime.utcnow()


## Задача 4. Конфигурация приложения (`BaseSettings`)

Опишите настройки подключения к внешнему API:

- `APISettings(BaseSettings)` с полями:
  - `base_url: HttpUrl`
  - `token: SecretStr`
  - `timeout_sec: int = 5` (1–60)
  - `retries: int = 2` (0–10)
- Используйте `model_config = ConfigDict(env_prefix="API_", env_file=".env", extra="ignore")`
- Проверьте, что значения корректно читаются из переменных окружения.

В тесте ниже среда заполняется вручную.


In [84]:
import os
from pydantic_settings import BaseSettings
from pydantic import ConfigDict, SecretStr, HttpUrl, field_validator

class APISettings(BaseSettings):
    # TODO: поля и валидации
    base_url: HttpUrl
    token: SecretStr
    timeout_sec: int = 5
    retries: int = 2


    # пример проверки диапазона для timeout_sec / retries
    @field_validator("timeout_sec", "retries")
    @classmethod
    def check_ranges(cls, v: int, info):
        if info.field_name == 'timeout_sec':
          if 1 <= v <= 60:
            return v

          raise Exception("timeout_sec should be 1<=x<=60")

        elif info.field_name == 'retries':
          if 0 <= v <= 10:
            return v
          raise Exception("retries should be 0<=x<=10")


    model_config = ConfigDict(
        env_prefix="API_", env_file=".env", extra="ignore"
    )


os.environ["API_BASE_URL"] = "https://google.com"
os.environ["API_TOKEN"] = "QERUrejgoejrgopejg23434"
os.environ["API_TIMEOUT_SEC"] = "10"
os.environ["API_RETRIES"] = "10"
api_settings = APISettings()

logger.info(f"APISettings instance: {api_settings}")

2025-11-23 13:42:51,997 - __main__ - INFO - APISettings instance: base_url=HttpUrl('https://google.com/') token=SecretStr('**********') timeout_sec=10 retries=10


## Задача 5. Извлечение из ORM (`from_attributes=True`)

Создайте простую SQLAlchemy-модель `SAUser(id, email, is_active)` (in-memory, без БД) и соответствующую модель Pydantic:

- Pydantic-модель `UserOut` с полями `id: UUID`, `email: EmailStr`, `is_active: bool`.
- Включите поддержку `from_attributes` в `model_config`.
- Создайте инстанс `SAUser` и провалидируйте его через `UserOut.model_validate(sa_user_instance)`.

Проверьте, что преобразование сработало.


In [85]:
from typing import Optional
from sqlalchemy import Column, String, Boolean
from sqlalchemy.orm import declarative_base
from uuid import uuid4
from pydantic import BaseModel, EmailStr, ConfigDict, ValidationError

Base = declarative_base()

class SAUser(Base):
    __tablename__ = "users"
    id = Column(String, primary_key=True, default=lambda: str(uuid4()))
    email = Column(String, nullable=False)
    is_active = Column(Boolean, default=True)

    def __init__(self, email: str, is_active: bool = True):
        self.id = str(uuid4())
        self.email = email
        self.is_active = is_active

class UserOut(BaseModel):
    # TODO: опишите поля и включите from_attributes
    id: UUID
    email: EmailStr
    is_active: bool

    model_config = ConfigDict(
        # TODO: включите режим атрибутов
        from_attributes=True
    )



sa_user = SAUser("filipp@mail.ru", True)
successfull_try = UserOut.model_validate(sa_user)
logger.info(f"successfull try: {successfull_try}")


sa_user_broke = SAUser("filippmail.ru", True)
try:
  bad_try = UserOut.model_validate(sa_user_broke)
except ValidationError as err:
  logger.error(f"bad try: {err}")


2025-11-23 13:42:54,508 - __main__ - INFO - successfull try: id=UUID('a969c03c-c8ee-4839-b1d8-ee8786b986b6') email='filipp@mail.ru' is_active=True
2025-11-23 13:42:54,509 - __main__ - ERROR - bad try: 1 validation error for UserOut
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='filippmail.ru', input_type=str]


## Задача 6. JSON Schema и дружелюбные ошибки

1. Для модели из задачи 3 сгенерируйте JSON Schema (метод `model_json_schema`) и запишите его в переменную `ORDER_SCHEMA`.
2. Реализуйте функцию `safe_create_order(data: dict) -> tuple[bool, str]`, которая:
   - пытается создать `Order` из входного `dict`,
   - при успехе возвращает `(True, "<total=...>")`,
   - при ошибке возвращает `(False, "<короткое сообщение об ошибке>")` без стек-трейса.

Не используйте сторонние библиотеки.


In [86]:
# Используем модели из задачи 3: OrderStatus, OrderItem, Order
from pydantic import ValidationError


ORDER_SCHEMA = Order.model_json_schema()

def safe_create_order(data: dict) -> tuple[bool, str]:
    # TODO: реализуйте безопасное создание заказа
    try:
      order = Order(**data)
      return True, f"<total={order.total}>"
    except ValidationError as err:
      return False, str(err)

successfull_try = safe_create_order({"id":"cad5a9b1-ab18-485a-8595-7ac1a844412c","user_email":"filipp@mail.com","items":[{"sku":"AAA","qty":2,"unit_price":"12.23"}],"status":"new","created_at":"2025-11-23T13:09:25.089715","total":"24.46"})
bad_try = safe_create_order({"id":"ad5a9b1-ab18-485a-8595-7ac1a844412c","user_email":"filipp@mail.com","items":[{"sku":"AAA","qty":2,"unit_price":"12.23"}],"status":"new","created_at":"2025-11-23T13:09:25.089715","total":"24.46"})

logger.info(f"successfull try: {successfull_try}")
logger.error(f"bad try: {bad_try}")

2025-11-23 13:42:57,264 - __main__ - INFO - successfull try: (True, '<total=24.46>')
2025-11-23 13:42:57,265 - __main__ - ERROR - bad try: (False, "1 validation error for Order\nid\n  Input should be a valid UUID, invalid group length in group 0: expected 8, found 7 [type=uuid_parsing, input_value='ad5a9b1-ab18-485a-8595-7ac1a844412c', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.11/v/uuid_parsing")
